<div class="alert alert-block alert-info">This Jupyter Notebook is part of the course <strong>"LLMs as Judges for Search" by OpenSource Connections.</strong>
    
Check out https://opensourceconnections.com/training/ for the full course and other classes.</div>

# Lab: Critique Model

In this lab, we will let the LLM compare the its own ratings with those of the human rater. We'll then ask it to provide an instruction that we can pass to its prompt and check if that increased the agreement with the human rater.


## Imports

In [2]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
import os
import openai
from openai import OpenAI
from sklearn.metrics import cohen_kappa_score


/opt/homebrew/lib/python3.10/site-packages/scipy/__init__.py:155: UserWarning: A NumPy version >=1.18.5 and <1.25.0 is required for this version of SciPy (detected version 1.25.2
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [3]:
load_dotenv(dotenv_path='../.env')

api_key = os.environ.get('OPENAI_API_KEY')

In [4]:
pd.set_option('display.max_colwidth', None)

## Product Data to Judge

Look at the following dataset. It contains search result data for the queries 'duct tape', 'iphone xr cool cases for teenage girls' and 'laptop' together with some human ratings that came from the ESCI dataset. How do they compare to the ratings that we'll get from the LLM?



In [5]:
df_products = pd.read_json('../data/esci-mini.json')

df_products.head(5)

,query,product_id,esci_label,binary_label,label,product_title,product_description,product_bullet_point,product_brand,product_color,product_locale
0,duct tape,B00DJWTGAG,E,1,3,"3M 2979 Multi-Use Duct Tape, Silver, 1.88 in x 60 yd x 7 mil, 1 Pack, Temporary Repair, Patching, Tabbing, Capping Pipe, Marking, Labeling",None,"Commercial grade – Silver duct tape resists curling and tears off roll cleanly for light use in professional MRO/construction applications\nFlexible adhesive – Aggressive synthetic rubber adhesive sticks immediately to a wide variety of surfaces\nMulti-use applications– Great for patching, tabbing, light duty bundling, capping pipe, marking, labeling or temporary repair\nPriced for economy – This multipurpose contractor-grade tape is economically priced to best provide a quality temporary solution for light duties",3M,Silver,us
1,duct tape,B078M21NYH,E,1,3,"Craftzilla Rainbow Colored Duct Tape — 6 Bright Colors — 10 Yards x 2 Inch — No Residue, Tear by Hand & Waterproof — Great for Arts & Crafts, Color-Coding, and DIY Projects","This multi purpose rainbow set of duct tape colors and patterns is great for Students, Teachers, Parents, Artists, and Professionals. Use this colored duct tape for crafts and art projects that can be done in the classroom, studio, kitchen, home, or garage. The colorful duct tape rips nicely and cleanly so it can easily be used and applied to different surfaces. This fun duct tape bulk pack comes with 6 fun duct tape craft rolls - each roll measuring 10 yards of 2 inch duct tape. This includes 6 rolls in an assortment of bright neon duct tape colors (not fluorescent) including: Pink, Orange, Yellow, Green, Blue, Purple. This colorful tape set can be made into DIY arts and crafts projects such as the construction of forts, making duct tape wallets, keychains, origami, personalizing and decorating journals, notebooks, and luggages. This is also great for color-coding, labeling, and organizing use making it a fun addition to your DIY kits. Whenever you move or need storage organization, use these duct tape assorted colors to make labels for boxes and storage bins and put corresponding duct tape rainbow color tags to each room. Discover creative possibilities with these duct tape arts and crafts kits. Art is limitless so use this duct tape for boys, girls, adults, artists, and professionals!","Vivid Vibrance – Enjoy eye-catching brightness and a full rainbow of color options. Make labels, gifts, and statements—your multi purpose color duct tape pack from Craftzilla is both your palette and canvas.\nTearable and Easy to Clean – Assemble rainbow duct tape kits at home, in school, and on vacation. Your craft tapes are easy for tiny hands to tear for their craft and construction projects, and no hassle to peel off for easy cleanup with no residue.\nYou’re on a Roll – And you’ve got plenty left! With 60 total yards of wonderful hues and easy-tear workability, you can use your colored duct tape variety pack for project after project.\nCommunicate With Color – Apply your multi color duct tape anywhere you need to be heard without saying a word. Effortlessly label moving boxes, organize unruly TV cabling, and mark social-distancing spots.\nStick With Us – Count on Craftzilla for colored tape duct so fun, bright, and inspiring you won’t want to put it down. Your tape set is backed by our commitment to your colorful and crafty success.",Craftzilla,"Rainbow - Pink, Orange, Yellow, Green, Blue, Violet",us
2,duct tape,B0021L9MVO,I,0,0,"Duck HD Clear Heavy Duty Packing Tape, 1.88 Inch x 109 Yards, 6 Rolls (299016)",None,"Heavy duty for a strong and secure hold to keep your valuables safe while moving, shipping or in storage\nOffers wide temperature range performance for shipping and storage in hot or cold temperatures\nAdhesive bond strengthens over time for a long-lasting hold on boxes, perfect for storage\nCrystal clear to the core for a professional look on boxes or taping address labels\nMeets postal

## Ratings without critique
This will serve as our baseline.

In [6]:
SYSTEM_PROMPT = """

You are an expert relevance judgment system. Your task is to assess the relevance of a given document to a specific user query.
The document is a product title.

In addition to the rating, provide a concise reasoning for your judgment.
First reason, then judge based on the reasoning.

Provide your relevance rating by assigning the label 'relevant' or 'not relevant' as property 'relevance' of an object in JSON format and the reasoning as property 'reasoning'.

"""

In [7]:
def make_user_prompt(query, doc_title):
    return f"""
    
User Query: {query}

Document title:\n{doc_title}

Based on the above, provide a relevance judgment."""



In [8]:
CLIENT = OpenAI(api_key=api_key)

def evaluate(query, doc_id, doc_title, response_judgment_property='relevance', response_reasoning_property='reasoning', client=CLIENT, system_prompt=SYSTEM_PROMPT):
    
    try:
        # Send request to OpenAI API
        # Using generate_content and specifying the response_mime_type for JSON output
       
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": system_prompt},
                {
                    "role": "user",
                    "content": make_user_prompt(query, doc_title),
                },
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        
        # Parse the JSON response
        json_output = response.choices[0].message.content
        judgment = json.loads(json_output)
        #print(judgment)
        
        return judgment[response_judgment_property], judgment[response_reasoning_property]
        
        

    except Exception as e:
        print(f"Error processing OpenAI judgment for Query: '{query[:30]}...', Doc ID: {doc_id}: {e}")
        raise e
        #return None
        

In [9]:
def evaluate_dataset(df, response_judgment_property='relevance', response_reasoning_property='reasoning', client=CLIENT, system_prompt=SYSTEM_PROMPT):
    df = df.copy()
    df[['judgment', 'reasoning']] = df.apply(lambda row: evaluate(query=row['query'], doc_id=row['product_id'], doc_title=row['product_title'], 
                                    response_judgment_property=response_judgment_property,
                                    response_reasoning_property=response_reasoning_property,
                                    client=client, system_prompt=system_prompt), axis=1, result_type="expand")
    df['binary_judgment'] = df['judgment'].apply(lambda j: 1 if j == 'relevant' else 0)

    df_kappa = df.groupby(by='query').apply(
        lambda g: cohen_kappa_score(g['binary_label'], g['binary_judgment'])
    ).to_frame(name='kappa').reset_index()
    return df, df_kappa

In [10]:
df_eval, df_kappa = evaluate_dataset(df_products)
df_kappa

,query,kappa
0,duct tape,0.600000
1,iphone xr cool cases for teenage girls,0.800000
2,laptop,0.411765


In [11]:
df_eval[df_eval['binary_judgment'] != df_eval['binary_label']][['query', 'product_title', 'judgment', 'binary_label']]

,query,product_title,judgment,binary_label
7,duct tape,"ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red",not relevant,1
9,duct tape,JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black),relevant,0
13,iphone xr cool cases for teenage girls,"iPhone Xs Case for Girls, YeLoveHaw Flexible Soft Slim Fit Full-Around Protective Cute Shell Phone Case Cover with Purple Floral and Gray Leaves Pattern for iPhone X/XS 5.8 Inch (Pink Flowers)",not relevant,1
20,laptop,Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo 15.6 Inch Intel Ci3 8GB 1TB Laptop,relevant,0
21,laptop,Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo (15.6 inch HD) Notebook (AMD A4-9125 2x2.6 GHz,relevant,0


## Applying critique about differences between human and LLM judgments
We now ask your LLM about the differences between human and LLM judgments and for an instruction for the LLM to behave like the human rater. 

In [12]:
CRITIQUE_SYS_PROMPT = """
You are an expert relevance judgment system. Your task is to review the work of relevance raters.
You will be given pairs of search queries and documents together with binary relevance ratings from two raters.

The relevance ratings come as labels 'relevant' or 'not relevant'. 

Think about what the two raters include in their idea of relevance and what not. 
Return your reasoning about these differences between the two raters as property 'analysis' and a very short summary as property 'summary' of a JSON object.

Based on that summary, also provide a very clear and concise instruction for rater 2 that makes it behave like rater 1 as property 'instruction' of that JSON object.
Note that rater 2 does not know about rater 1.


"""

This is what the output of the `make_critique_user_prompt` function below could look like.

```

---

User Query: duct tape

Document title: ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red

Rater 1: relevant

Rater 2: not relevant

---

User Query: duct tape

Document title: JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black)

Rater 1: not relevant

Rater 2: relevant

---

User Query: laptop

Document title: Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo 15.6 Inch Intel Ci3 8GB 1TB Laptop

Rater 1: not relevant

Rater 2: relevant

---

User Query: laptop

Document title: Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo (15.6 inch HD) Notebook (AMD A4-9125 2x2.6 GHz

Rater 1: not relevant

Rater 2: relevant

---

Based on the above, provide an analysis of the differences between the two raters.


```

In [13]:


def make_critique_user_prompt(df):
    
    prompt = '---'
    
    queries, titles, binary_labels, binary_judgments = df['query'], df['product_title'], df['binary_label'], df['binary_judgment']
    
    for query, title, binary_label, binary_judgment in zip(queries, titles, binary_labels, binary_judgments):
        
        prompt += f"""
User Query: {query}

Document title: {title}

Rater 1: {'relevant' if binary_label == 1 else 'not relevant'}

Rater 2: {'relevant' if binary_judgment == 1 else 'not relevant'}

---

        """
        
    prompt += 'Based on the above, provide an analysis of the differences between the two raters.'
    
    #print(prompt)
    
    return prompt
    
    

In [14]:
def get_analysis(df_diff):
    try:
        # Send request to OpenAI API
        # Using generate_content and specifying the response_mime_type for JSON output
       
        response = CLIENT.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": CRITIQUE_SYS_PROMPT},
                {
                    "role": "user",
                    "content": make_critique_user_prompt(df_diff),
                },
            ],
            response_format={"type": "json_object"},
            temperature=0.0
        )
        
        # Parse the JSON response
        json_output = response.choices[0].message.content
        critique = json.loads(json_output)
        
        
        return critique['analysis'], critique['summary'], critique['instruction']
        
       
        
        

    except Exception as e:
        
        raise e
        #return None

In [15]:
df_diff = df_eval[df_eval['binary_judgment'] != df_eval['binary_label']]
analysis, summary, instruction = get_analysis(df_diff)

print(f"Analysis: {summary}")
print(f"Summary: {summary}")
print(f"Instruction: {instruction}")

Analysis: Rater 1 is stricter about direct relevance to the query, while Rater 2 is more lenient and considers broader associations.
Summary: Rater 1 is stricter about direct relevance to the query, while Rater 2 is more lenient and considers broader associations.
Instruction: Focus on the direct relevance of the document to the specific query, ensuring that the content closely matches the intent or subject of the query.


## Extending the prompt with the instruction


In [16]:
SYSTEM_PROMPT_WITH_ADDED_INSTRUCTION = f"""
You are an expert relevance judgment system. Your task is to assess the relevance of a given document to a specific user query.
The document is a product title.

{instruction} - follow this instruction closely.

In addition to the rating, provide a concise reasoning for your judgment.
First reason, then judge based on the reasoning.

Provide your relevance rating by assigning the label 'relevant' or 'not relevant' as property 'relevance' of an object in JSON format and the reasoning as property 'reasoning'.
"""

In [17]:
df_eval, df_kappa = evaluate_dataset(df_products, system_prompt=SYSTEM_PROMPT_WITH_ADDED_INSTRUCTION)
df_kappa

,query,kappa
0,duct tape,0.600000
1,iphone xr cool cases for teenage girls,0.800000
2,laptop,0.736842


In [18]:
df_eval[df_eval['binary_judgment'] != df_eval['binary_label']][['query', 'product_title', 'judgment', 'binary_label', 'reasoning']]

,query,product_title,judgment,binary_label,reasoning
7,duct tape,"ChromaLabel 1 Inch Clean Remove Color-Code Tape, 500 Inch Roll, Red",not relevant,1,"The document title refers to color-code tape, which is not the same as duct tape. Duct tape is typically a stronger, more versatile adhesive tape, while the product mentioned is a specific type of tape designed for color coding and clean removal, which does not match the user's query."
9,duct tape,JVCC J90 Low Gloss Gaffer-Style Duct Tape: 4 in. (96mm Actual) x 75 ft. (Black),relevant,0,"The document title includes 'duct tape' and specifies a product that is a type of duct tape, which directly matches the user query."
13,iphone xr cool cases for teenage girls,"iPhone Xs Case for Girls, YeLoveHaw Flexible Soft Slim Fit Full-Around Protective Cute Shell Phone Case Cover with Purple Floral and Gray Leaves Pattern for iPhone X/XS 5.8 Inch (Pink Flowers)",not relevant,1,"The document title refers to a case for the iPhone Xs, not the iPhone XR, which is specifically mentioned in the user query. Additionally, while it is targeted towards girls, the design and features do not align closely with the query's emphasis on 'cool cases' for teenage girls."
21,laptop,Broonel Black Invisible Lightweight Laptop Computer Stand - Compatible with The Lenovo (15.6 inch HD) Notebook (AMD A4-9125 2x2.6 GHz,relevant,0,"The document title mentions a 'Laptop Computer Stand' which is directly related to laptops. It also specifies compatibility with a Lenovo notebook, indicating a clear connection to the user query about laptops."
